##  Modelling of flood damages with Non-parametric Bayesian Network

In this exercise, we apply a non-parametric Bayesian Network (BN) on a flood damage dataset about the impacts of a severe flooding in the Netherlands in the 1990s.

The structure of a BN enables to predict even more than a single variable, i.e. it is a probabilistic approach. We make use of this by calibrating the model on predicting content damages (damages to the inventory such as furniture or electrical devices). 
In the BN graph we can set the conditional dependencies based on logical conclusions (and expert knowledge) between the candidate predictors themselves as well between predictors and the target (`Content damage`).
> For instance, a conditional dependency exist between: `Structure damage`--> `Content damage`\
Because flood water infiltrates first through walls and openings of doors and windows leading to structural damages at the building (`Structure damage`), before the water level in the building rises causing damages to devices and furniture in the rooms (`Content damage`).


### Description of the variables in the flood damage dataset 

- `Structure damage`: The damage costs to buildings [unit: Guilder, price level 1993]
- `Content damage`: The damage costs to the inventory (e.g. to furniture, electrical devices etc.), we try to predict this variable [unit: Guilder, price level 1993]
- `Water depth`: Water level inside the building, relative to the floor level [meter]
- `Flow velocity`: The "speed" of the water in front of the building [m/s]
- `Duration`: Duration expressing how long each building is inundated [h]
- `Inhabitants`: Number of people living in each building [n]
- `Basement`: Building has a basement [1= yes, 2= no]
- `Detached`: It is a detached building [1= yes, 2= no]
- `footprint`. Footprint are of the building [m²]
- `Construction age`: Building age [year]
- `Living area`: Floor area for living [m²]
- `Return period`: Represents at which return period the building is first floods [year]



BN package: https://github.com/mike-mendoza/py_banshee


**Important hint**:
> Use the [Quick_start_guide_Py_BANSHEE](https://github.com/mike-mendoza/py_banshee/blob/main/Quick_start_guide_Py_BANSHEE.pdf) for information about the py_banshee functions

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

from py_banshee.rankcorr import bn_rankcorr
from py_banshee.bn_plot import bn_visualize
from py_banshee.copula_test import cvm_statistic
from py_banshee.d_cal import gaussian_distance
from py_banshee.prediction import inference


seed = 42

## Download the data
Download the zip file `07_data.zip` from this [folder](https://tubcloud.tu-berlin.de/s/ZX6LbyAQzC5i6RL).
Unzip the folder and place it in a folder called `./data` in the same directory of this exercise.


## Define some additional functions for model evaluation

In [ ]:
def mean_bias_error(y_true, y_pred):
    """Calculate MBE from predicted and actual target"""
    return (y_pred - y_true).mean()


def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    """Calculate SMAPE from predicted and actual target"""
    return 100 / len(y_true) * np.sum(np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true)))


def empirical_vs_predicted(y_true, y_pred):
    """
    return (pd.DataFrame): with statistics of predicted and observed target values
    """
    empirical_vs_predicted = []

    for y_set in [y_true.astype(int), y_pred.astype(int)]:
        test_statistics = stats.describe(np.array(y_set))

        empirical_vs_predicted.append(
            pd.Series(
                {
                    "nobs": test_statistics[0],
                    "median": np.median(y_set),
                    "mean": np.mean(y_set),
                    "min max": [test_statistics[1][0], test_statistics[1][1]],
                    "variance": round(test_statistics[3], 2),
                    "standard deviation": round(np.std(y_set), 2),
                }
            )
        )
    return pd.DataFrame(empirical_vs_predicted, index=(["empirical", "predicted"]))




In [ ]:
# Define a function for plotting observed vs predicted values

from matplotlib.colors import to_rgba

def plot_observed_predicted(
    y_true, y_pred, hue=None, hue_colors=("darkgrey", "steelblue"), xlabel="observed", ylabel="predicted", alpha=0.6, legend=False, outfile="test.png"
):
    """
    # Code Snippet: https://stackoverflow.com/questions/66667334/python-seaborn-alpha-by-hue
    """

    sns.set(style="white", font_scale=1.2)

    color_dict = {
        0: to_rgba(hue_colors[0], alpha),  # set transparency for each class independently
        1: to_rgba(hue_colors[1], alpha),
    }
    if hue is None:
        color_dict = color_dict[1]

    g = sns.JointGrid(
        x=y_true,
        y=y_pred,
        hue=hue,
        height=5,
        space=0,
    )
    p = sns.scatterplot(x=y_true, y=y_pred, hue=hue, palette=color_dict, edgecolors=color_dict, legend=legend, ax=g.ax_joint)

    if legend is True:
        plt.setp(p.get_legend().get_texts(), fontsize="12")
        plt.setp(p.get_legend().get_title(), fontsize="15")

    g.plot_marginals(sns.histplot, kde=True, palette=color_dict, fill=True)  # multiple='stack')

    g1 = sns.regplot(x=y_true, y=y_pred, line_kws={"lw": 1.0}, scatter=False, ax=g.ax_joint)
    regline = g1.get_lines()[0]
    regline.set_color("steelblue")

    x0, x1 = (0, 100)
    y0, y1 = (0, 100)
    lims = [min(x0, y0), max(x1, y1)]
    g.ax_joint.plot(
        lims,
        lims,
        c="black",
        lw=0.5,
    )  # equal line
    g.set_axis_labels(xlabel=xlabel, ylabel=ylabel)

    # # save plot
    # plt.savefig(outfile, dpi=300, bbox_inches="tight")

    plt.show()

## Load the data

### Task:
* Load the `building_damages_v2.csv` .
* Replace all missing observations (-999) with `numpy.nan` .

## Convert Guilders to Euros

### Task: 
* Convert the dutch currency [Guilders] to Euros for a better readability.

> Note that in 1993 (the year when the dataset was created), the official exchange rate that was eventually used for the Euro conversion in 1999 (and adopted in 2002) was not yet in place. However, the Dutch guilder was pegged to the "Deutsche Mark", and based on the final, fixed conversion rate of 2.20371 Guilder to 1 Euro.

For the exercise, we assume an exchange rate of 2.20371


In [ ]:
## conversion of Guilders -> EURO at price level of 1993



## Exploratory analysis of the target variable

### Task: 
* Plot the distribution of the target variable for content damages. Which distribution due you see and what impact would you assume for the model?
* Also, explore the (unconditional) correlations between the variables and find the predictor with the strongest correlation to the target variable (e.g. use the seaborn `heatmap()`and the spearman rank correlation)
* Which is the predictor with the highest unconditional correlation to the target?

# First calibration round

## Construct the Bayesian Network graph

### Task:
Define the variable names in the list. 
Start first with the target variable and the most important predictor (based on correlation plot from before). In our case, we have two most important predictors, so we start either with both of them.
Then try out subsequently further predictors and see how the performance of the BN changes.
Keep only a predictor if it improves the performance of the BN and has a conditional correlation of > 0.10 to another variable.
Otherwise remove it again from the list. 


In [ ]:
## Example (starting point)


# names_list = ["Content damage", "Structure damage",  "Water depth", ]

# # # Select the columns to use in the NPBN
# df_bn = df_data[names_list]

# names = {k: i for i, k in enumerate(names_list)}

# print("Using following features:\n", names)

### Task:
Create a Directed Acyclic Graph (DAG). 
The DAG is a modeling structure composed of nodes and directed edges, where connections follow a one-way flow without forming loops (cycles).
* Extract the number of nodes for the DAG  (number of nodes = number of features + target), start with the three variables from before.
* Define the structure of the DAG


In [ ]:
## Example

# # Extract number of nodes  (number of nodes = number of features + target)
# N = df_bn.shape[1] 
# print(N)

# # Defining the structure of the BN
# parent_cell = [None]*N
# parent_cell[0] = [1, 2]  #   target <-  (parents: structure damage, water depth)
# parent_cell[1] = []  #   structure damage (parent: none)
# parent_cell[2] = [1]      # water depth in the building <- (parent: structure damage) 


# parent_cell

Task:
* Plot the conditional correlations use for this the function: `bn_rankcorr`
* Explore the conditional correlations in the BN graph (based on spearman rank correlation)

## Calibrate and evaluate the BN 
Apply the two code cells below. In the code cells below, we conduct the following:
* 1st cell: Get information about the different input parameters of the py banshee functions.
* We use for evaluation a simple 5-fold cross-validation (train/test ratio = 80 / 20 ) and assess the BN performance based on MAE, RMSE, SMAPE and MBE. The two last metrics are less common (but in my opinion quite useful). 
    * Conduct a small research about the benefits and disadvantages of all four performance metrics
* Conditionalize the graph of the BN on the candidate predictors (here: structure damage, water depth) to inference on the remaining node(s) (here: content damage)
* Check the output of `Averaged evaluation scores of k-fold cross-validation (mean) :` 

**Note**
* In the code cell below, we are drawing 1000 samples from our distributions during each CV fold, which means some records in the test dataset are taken multiple times (size test dataset per CV fold: ~ 879 records). 
* The test data (per CV fold) is used for predicting the unconditionalized nodes (here: content damage), see function `inference()`. We get for each iteration a new predicted distribution of the content damage variable. This is because we are taking each time another CV fold as test set and draw randomly from it 1000 samples to conduct the inference. 
* We get as output of `inference()` a probability distribution of the predicted content damage (for each CV fold). However, this distribution is less useful when we want to apply our performance metrics (MAE, RMSE etc.). Thus, we are taking the `mean` of the predicted distributions of content damage and then apply our performance measures (during each CV fold)

In [ ]:
# ## get information about the input parameters of the pybanshee functions.
# # inference?

# Example for rank correlation matrix and how to conduct the inference

# # test dataset containing the data used to conduct inference (i.e. prediction), it is similar to "X_test"
# values = test_nth_fold.iloc[:,condition].to_numpy() 

# # BN get rank correlations per fold
# R = bn_rankcorr(
#     parent_cell = parent_cell,        # structure of the BN
#     data = train_nth_fold,
#     var_names = names,  # names of variables
#     is_data = True,        # matrix data contains actual data
#     plot = False
# )  

# # make inference on the unconditionalized node(s) (i.e., the target variable)
# F_test_nth_fold = inference(
#     Nodes= condition,        # nodes that will be conditionalized (i.e. our features, similar to the columns from "X_test")
#     Values = values,           # test dataset contains the data with which the inference (i.e. prediction of content damage) should be conducted (similar to "X_test")
#     R = R,                # conditional correlations
#     DATA = test_nth_fold,
#     SampleSize = 1000,  
#     Output = "mean"    # type of output data, e.g. mean of the predicted distribution(s) of unconditionalized node(s)
# )


Write down a few words about the evaluation results:

### Plot observations vs predictions

In [ ]:
# Example 

# ## add residuals
# residuals["residual"] = residuals["y_pred"] - residuals["y_true"]

# plot_observed_predicted(
#     residuals["y_true"], residuals["y_pred"],  
#     xlabel=f"Observed content damages ", ylabel=f"Predicted content damages ",
#     legend=False,
# )   

What do we see?


# 2. Calibration round  (Intermediate and final version of the DAG structure)

### Task: 
Add further candidate predictors to the DAG and evaluate the model with the new features.
1. Re-run the workflow multiple times each time test one further candidate predictor (e.g., add the `flow velocity` as parent node of `water depth`) or change the dependencies in the DAG (see, the code part below `Defining the structure of the BN`)
2. Check the conditional correlation matrix if the correlation to another variable is at least medium strong (> 0.10) 
3. If point "2." is fulfilled, rerun the cross validation to evaluate the performance of the BN. Did the new feature in DAG or change in the dependency structure improve the model performance?


In [ ]:
#1

In [ ]:
# 2

In [ ]:
3.

Did the model improved with the new features and their defined dependencies?

## Visualize the BN

Lets plot he Bayesian Network with the function `bn_visualize()`


In [ ]:
# In case you get an ExecutableNotFoundError (for linux):
# !sudo apt-get install graphviz 


The plot presents the BN with x nodes and x arcs, with the (conditional) rank correlations indicated on the arcs.


Lets plot the marginal distributions.
Remember from the lecture session that a marginal distribution is where you look only on the distribution of one of two variables (e.g. either X or Y) in a probability table, ignoring the impacts of other variables. In other words, a marginal distribution is the sum probabilities of a variable of interest, independently from others.



## Copula test

Actually we would now test our assumption that a Gaussian copula (as we assumed above) is well suited to model the bivariate dependencies between the variables in the BN model. However, we skip this step for now (bc I got an TypeError for this function and I have not spend some time to find the solution for this dataset ;) )
> TypeError: Invalid value for dtype 'str'. Value should be a string or missing value (or array of those).



**Without the TypeError, we would do the following:**

> From the `Quick start guide`: \
 *The function cvm_statistic computes the sum of squared differences between the parametric and empirical copulas. The lower value of the resulting M metric, the better fit between the parametric and empirical copulas is achieved. The only required argument is the matrix of data used in the previous steps* 

> **Task:**
>* Use the `cvm_statistic()` to conduct a goodness-of-fit test on the entire dataset
> * Assess if the parametric copulas (in the form of Gaussian copulas) are well representing the empirical copulas. The latter are based on our loss data, the former ones are the assumed Gaussian copulas

```
# Goodness-of-fit test for the Gaussian copula 
plt.figure(figsize=(20,15))
M = cvm_statistic(
    DATA=df_bn,             # Dataframe
    names = names , #df_bn.columns,  # names of variables
    plot = True,               # create a plot (0=don"t create plot)
    fig_name = "goodness_of_fit_test"    # figure name
)
```

Goodness-of-fit test:

The results of the goodness-of-fit test in terms of Cramer-von Mises statistic highlight that the Gaussian copula is in the majority of cases the most suitable one for representing the dependencies between variables, especially for the variables of interest (building and content damages). This is important as the method utilizes the Gaussian copula for dependence modelling.



## Uncertainty quantification

### Task:
Conduct an uncertainty quantification by using the `bn_rankcorr()` and `inference` functions from before.
* You can simply reapply both functions from before, just simply return the output of `bn_rankcorr()` as np.array (i.e. `plot=False`). Use as input the entire dataset. 
* Uncomment the code in the code cell below, which will give you the values of the conditionalized variables (i.e. the predictors). You need this variable for the parameter "Values" in `inference()`. Additionally, set in `inference()` the `Output` parameter to `"full"` which will return for each predicted value of `Content damage` a probability distribution. In other words, we get 4397 probability distributions for `Content damage`



In [ ]:
## Example 

# values = df_bn.iloc[:,condition].to_numpy() # data for conditionalization

# # BN define rank corr coefs on HCMC
# R = bn_rankcorr(
#     parent_cell,        # structure of the BN
#     df_bn,  
#     var_names = names,  # names of variables
#     is_data = True,        # matrix data contains actual data
#     plot = False
# )  

# ## BN predict on Can Tho 
# F = inference(
#     Nodes = condition,        # nodes that will be conditionalized
#     Values = values,           # information used to conditionalize the nodes of the NPBN
#     R = R,                # the rank correlation matrix from HCMC ds
#     DATA = df_bn,        # DataFrame for cantho
#     SampleSize = 1000, 
#     Output = "full"    # type of output data
# )


Uncomment the code cell below to extract the values of the probability distributions ("F"), called y_pred. In detail, each probability distribution consists of 1000 samples.\
The empirical data about the content damages is stored in y_true

In [ ]:
## predict target and observed target
# y_pred = F.squeeze()
# y_true = df_bn.iloc[:,0].to_numpy()

# print(y_pred.shape)
# print(y_true.shape)


Uncomment the two code cells below to average across the 1000 samples

In [ ]:
# ## average uncertainties

# dict_uncertainties = {}

# for i in range(0,1000):
#     nth_sample  = [ e[i] for e in y_pred ]
#     dict_uncertainties[i] = nth_sample

In [ ]:
# df_uncertainties = pd.DataFrame.from_dict(dict_uncertainties).T  # cols: samples or flood cases, rows: sample number 
# df_uncertainties.tail(10)


# # get mean of uncertainty distbution for each flood case
# df_uncertainties_avg = df_uncertainties

# df_uncertainties_p = pd.DataFrame()
# df_uncertainties_p["avg_modelled"] = df_uncertainties_avg.mean(axis=0).round(0)
# df_uncertainties_p["observed"] = y_true
# df_uncertainties_p["region"] = "Meuse region"  # or "test" if you want to distinguish between training and test set

# df_uncertainties_p



### Plot model uncertainty

We are making it easy - we simply apply the pre-written code in the cell below

In [ ]:
# group = "region"
# column = "avg_modelled"
# scatterpoints="observed"


# grouped = df_uncertainties_p.groupby(group)
# categories = np.unique(df_uncertainties_p[scatterpoints])
# colors = np.linspace(0, 1, len(categories))


# names, vals, xs, colrs = [], [] ,[], []

# for i, (name, subdf) in enumerate(grouped):
#     names.append(name)
#     vals.append(subdf[column].tolist())
#     xs.append(np.random.normal(i+1, 0.04, subdf.shape[0]))
#     colrs.append(subdf[scatterpoints].tolist())

# fig, ax = plt.subplots(figsize=(5, 5))
# fig.canvas.draw()


# ## average predicted means
# p = sns.boxenplot(
#     x = df_uncertainties_p["region"],
#     y = df_uncertainties_p["avg_modelled"],
#     width=.5,
#     showfliers=False,
#     palette=["teal"],
#     hue=df_uncertainties_p["region"],
#     line_kws={"linewidth":1.5},
#     flier_kws={"facecolor":.7, "linewidth":.5},
#     legend=False,
# )

# # observed mean
# sns.boxplot(
#     x = df_uncertainties_p["region"],
#     y = df_uncertainties_p["observed"],
#     showfliers=False,
#     showmeans=True,
#     medianprops={"color": "r", "linewidth": 0, "alpha":0.0},
#     boxprops={"facecolor":"steelblue", "alpha":0.0},
#     whiskerprops={"color":"k", "alpha":0.},
#     capprops={"color":"k", "alpha":0},
#     meanprops={"marker":"s", "markerfacecolor":"blue", "markeredgecolor":"blue"}
# )

# p.tick_params(bottom=False)  # remove x ticks
# plt.legend().set_visible(False)  # suppress legend

# plt.ylim(0, 15000)
# plt.ylabel(f"Modelled content damage [Euros]")
# plt.xlabel(f"")
# plt.title("Uncertainty range")

# plt.savefig(f"./uncertainties.png", dpi=300, bbox_inches="tight")


## Optional: Cumulative Distribution Function (CDF)

Lets visualize the prediction bias (i.e. cumulative error) for Meuse dataset as a CDF


In [ ]:
# residuals["residual"] = residuals["y_pred"] - residuals["y_true"]

In [ ]:

# sns.set(rc={"xtick.bottom" : True, "ytick.left" : True})
# sns.set_style(
#     style="ticks",
#      rc={"axes.grid" : False, "grid.linestyle": ":"}
#      )

# fig, ax = plt.subplots(1, 1, figsize=(5,5))

# p, bins, patches  = ax.hist(
#     [residuals["residual"]],  
#     bins=200,
#     density=True, # alias for normalize, last bins equals 1
#     cumulative=True,  
#     histtype='step', 
#     alpha=0.6,
#     color=("teal"), 
#     label=( "CDF Meuse"),
# )

# ax.grid( which='major', color='grey', linewidth=0.5)

# ## entire CDF
# plt.xlim(-15000, 10001)
# plt.xticks(np.arange(-15000, 10001, 5000), minor=False)
# plt.xticks(np.arange(-15000, 10001, 1000), minor=True)


# plt.xlabel("Prediction bias [Euros]")
# plt.ylabel("Probability")
# plt.legend(loc='best')

# plt.vlines(x=0, ymin=0, ymax=1.01, colors="black", linestyles="--")

# plt.title("CDFs of BN model", fontweight="bold", fontsize=16)
# plt.savefig("./cdf_bn_rbred.png", dpi=300, bbox_inches="tight")



